# 🎬 AI Shorts Factory (Auto-Generated)

This notebook automates the creation of viral YouTube Shorts using Reddit stories (or AI-invented ones) and background footage.

### Instructions:
1.  **Add your Gemini API Key:** In the Colab sidebar, go to 'Secrets' (key icon), add a new secret named `GEMINI_API_KEY` with your key, and enable notebook access.
2.  **Run All Cells:** The script will handle dependencies, content generation, and video rendering.
3.  **Output:** Find your videos in the `final_shorts` folder.

In [ ]:
# @title 1. Setup & Installation
# Install necessary libraries
!pip install moviepy edge-tts requests yt-dlp imageio-ffmpeg

# Fix ImageMagick policy for MoviePy TextClip on Linux/Colab
!apt install imagemagick fonts-liberation
!sed -i 's/none/read,write/g' /etc/ImageMagick-6/policy.xml

print("✅ Environment Setup Complete.")

In [ ]:
# @title 2. The AI Shorts Factory Script
import os
import time
import json
import random
import subprocess
import requests
from typing import List, Dict

try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except ImportError:
    # Fallback for local testing or if userdata is not set
    GEMINI_API_KEY = "YOUR_GEMINI_API_KEY_HERE" # Replace if running locally

if GEMINI_API_KEY == "YOUR_GEMINI_API_KEY_HERE" or not GEMINI_API_KEY:
    print("⚠️ WARNING: GEMINI_API_KEY not found in Secrets. AI generation may fail.")

# --- CONFIGURATION ---
SUBREDDIT = "confessions"
VOICE = "en-US-ChristopherNeural"
BACKGROUND_VIDEO_FILE = "background.mp4"
# A default satisfying video (Minecraft Parkour or similar royalty free style)
# This is a fallback URL if no local file is provided.
DEFAULT_YOUTUBE_URL = "https://www.youtube.com/watch?v=intRX7BRA90" 
OUTPUT_DIR = "final_shorts"
METADATA_FILE = "upload_metadata.json"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- 1. CONTENT ENGINE ---

def get_reddit_stories(limit=5) -> List[Dict]:
    """Attempts to fetch top stories from Reddit."""
    print(f"🔍 Attempting to scrape r/{SUBREDDIT}...")
    url = f"https://www.reddit.com/r/{SUBREDDIT}/top.json?t=day&limit={limit*2}"
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'}
    
    try:
        resp = requests.get(url, headers=headers, timeout=10)
        if resp.status_code == 200:
            data = resp.json()
            posts = []
            for child in data['data']['children']:
                p_data = child['data']
                if not p_data['over_18'] and len(p_data['selftext']) > 200: # Simple filters
                    posts.append({
                        "title": p_data['title'],
                        "script": p_data['selftext'][:800], # Truncate for length
                        "source": "reddit"
                    })
            print(f"✅ Fetched {len(posts)} stories from Reddit.")
            return posts[:limit]
        else:
            print(f"❌ Reddit blocked request (Status {resp.status_code}). Switching to AI Mode.")
            return []
    except Exception as e:
        print(f"❌ Reddit scrape error: {e}. Switching to AI Mode.")
        return []

def generate_ai_stories(count=5) -> List[Dict]:
    """Uses Gemini to invent spicy stories if Reddit fails."""
    print("🤖 Entering AI Invention Mode (Gemini Director)...")
    
    prompt = f"""
    You are a creative writer for viral YouTube Shorts. 
    Write exactly {count} distinct, dramatic, and engaging short stories (confessions, revenge, drama).
    Each story must be around 140 words (approx 40 seconds spoken).
    
    Return ONLY a raw JSON Array with this structure:
    [
      {{
        "title": "Hook/Title here",
        "script": "The full story text here...",
        "description": "YouTube description with hashtags",
        "tags": ["tag1", "tag2"]
      }}
    ]
    """
    
    url = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?key={GEMINI_API_KEY}"
    headers = {'Content-Type': 'application/json'}
    data = {"contents": [{"parts": [{"text": prompt}]}]}
    
    try:
        response = requests.post(url, headers=headers, json=data)
        result = response.json()
        text_content = result['candidates'][0]['content']['parts'][0]['text']
        
        # Clean up code blocks if present
        if "```json" in text_content:
            text_content = text_content.split("```json")[1].split("```")[0]
        elif "```" in text_content:
             text_content = text_content.split("```")[1].split("```")[0]
             
        stories = json.loads(text_content)
        print(f"✅ Gemini invented {len(stories)} stories.")
        return stories
    except Exception as e:
        print(f"❌ Critical Gemini Error: {e}")
        print(f"Response dump: {response.text if 'response' in locals() else 'No response'}")
        return []

# --- 2. MEDIA ENGINE ---

def ensure_background_video():
    """Checks for local file or downloads from YouTube."""
    if os.path.exists(BACKGROUND_VIDEO_FILE):
        print("✅ Local background.mp4 found.")
        return True
    
    print("📥 background.mp4 not found. Attempting YouTube download...")
    try:
        import yt_dlp
        ydl_opts = {
            'format': 'bestvideo[ext=mp4]+bestaudio[ext=m4a]/best[ext=mp4]/best',
            'outtmpl': 'background.%(ext)s',
            'quiet': False,
        }
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([DEFAULT_YOUTUBE_URL])
        
        if os.path.exists(BACKGROUND_VIDEO_FILE):
            print("✅ Download successful.")
            return True
        else:
            print("❌ Download seemingly succeeded but file not found.")
            return False
            
    except Exception as e:
        print(f"❌ YouTube Download Failed (likely Colab IP ban): {e}")
        print("⚠️ PLEASE MANUALLY UPLOAD a video named 'background.mp4' to the Colab file browser.")
        return False

def generate_audio(text, index):
    """Generates TTS audio using edge-tts CLI (subprocess)."""
    output_file = f"temp_audio_{index}.mp3"
    # Remove newlines to prevent CLI errors
    clean_text = text.replace("\n", " ").replace('"', "")
    
    cmd = ["edge-tts", "--voice", VOICE, "--text", clean_text, "--write-media", output_file]
    try:
        subprocess.run(cmd, check=True)
        return output_file
    except subprocess.CalledProcessError as e:
        print(f"❌ TTS Error: {e}")
        return None

def create_short(script_data, index):
    """Assembles the video using MoviePy."""
    from moviepy.editor import VideoFileClip, AudioFileClip, TextClip, CompositeVideoClip
    
    try:
        audio_path = generate_audio(script_data['script'], index)
        if not audio_path:
            return None
            
        audio_clip = AudioFileClip(audio_path)
        duration = audio_clip.duration
        
        # Load background
        video = VideoFileClip(BACKGROUND_VIDEO_FILE)
        
        # Loop if video is shorter than audio
        if video.duration < duration:
            video = video.loop(duration=duration + 1)
            
        # Random start point for variety
        max_start = max(0, video.duration - duration - 1)
        start_time = random.uniform(0, max_start)
        video = video.subclip(start_time, start_time + duration)
        
        # Crop to 9:16 Vertical
        # Assuming 1080p source usually, target 608x1080 or similar ratio
        w, h = video.size
        target_ratio = 9/16
        target_w = int(h * target_ratio)
        
        if target_w > w:
            # If video is too narrow, just center crop what we can or resize
            # But usually backgrounds are 16:9 landscape
            video = video.resize(height=1920)
            w, h = video.size
            target_w = int(h * target_ratio)
            
        crop_x1 = (w / 2) - (target_w / 2)
        crop_x2 = (w / 2) + (target_w / 2)
        
        video = video.crop(x1=crop_x1, y1=0, x2=crop_x2, y2=h)
        video = video.resize( (1080, 1920) ) # Force standard short res
        
        # Add Title Overlay (Center, Bold White)
        # Using TextClip
        # Note: 'size' is deprecated in some versions, 'fontsize' is standard
        title_text = script_data['title'].upper()
        # Using Liberation-Sans-Bold for better Linux/Colab compatibility
        txt_clip = (TextClip(title_text, fontsize=70, color='white', font='Liberation-Sans-Bold', method='caption', size=(900, None), stroke_color='black', stroke_width=2)
                    .set_position('center')
                    .set_duration(duration))
        
        final = CompositeVideoClip([video, txt_clip])
        final.audio = audio_clip
        
        output_filename = f"{OUTPUT_DIR}/short_{index+1}.mp4"
        # threads=1 is crucial for Colab to avoid memory crashes
        final.write_videofile(output_filename, codec='libx264', audio_codec='aac', threads=1, fps=24, logger=None)
        print(f"✅ Rendered: {output_filename}")
        
        # Cleanup temp audio
        audio_clip.close()
        video.close()
        os.remove(audio_path)
        
        return output_filename
        
    except Exception as e:
        print(f"❌ Video Render Error for #{index}: {e}")
        return None

# --- 3. MAIN EXECUTION ---

def main():
    print("🚀 Starting AI Shorts Factory...")
    
    # 1. Get Background
    if not ensure_background_video():
        print("🛑 Halting: No background video available.")
        return

    # 2. Get Content
    stories = get_reddit_stories(limit=5)
    if not stories:
        stories = generate_ai_stories(count=5)
    
    if not stories:
        print("🛑 Halting: No content generated.")
        return

    print(f"🎬 Processing {len(stories)} videos...")
    
    metadata = []
    for i, story in enumerate(stories):
        print(f"\n--- Generating Video {i+1}/{len(stories)}: {story['title']} ---")
        path = create_short(story, i)
        if path:
            metadata.append({
                "file": path,
                "title": story['title'],
                "description": story.get('description', f"{story['title']} #shorts"),
                "tags": story.get('tags', ["shorts", "reddit"])
            })
    
    # Save Metadata
    with open(METADATA_FILE, 'w') as f:
        json.dump(metadata, f, indent=2)
        
    print("\n✨ All Done! Check the 'final_shorts' folder.")

if __name__ == "__main__":
    main()